In [1]:
import sqlite3
import pandas as pd

DB_PATH = "chinook.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [2]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in database:")
for table in tables:
    print("-", table[0])

Tables in database:
- Album
- Artist
- Customer
- Employee
- Genre
- Invoice
- InvoiceLine
- MediaType
- Playlist
- PlaylistTrack
- Track


In [3]:
cursor.execute("PRAGMA table_info(Track);")
track_schema = cursor.fetchall()

print("Track table schema:")
for col in track_schema:
    print(col)

Track table schema:
(0, 'TrackId', 'INTEGER', 1, None, 1)
(1, 'Name', 'NVARCHAR(200)', 1, None, 0)
(2, 'AlbumId', 'INTEGER', 0, None, 0)
(3, 'MediaTypeId', 'INTEGER', 1, None, 0)
(4, 'GenreId', 'INTEGER', 0, None, 0)
(5, 'Composer', 'NVARCHAR(220)', 0, None, 0)
(6, 'Milliseconds', 'INTEGER', 1, None, 0)
(7, 'Bytes', 'INTEGER', 0, None, 0)
(8, 'UnitPrice', 'NUMERIC(10,2)', 1, None, 0)


In [4]:
cursor.execute("PRAGMA table_info(Genre);")
genre_schema = cursor.fetchall()

print("Genre table schema:")
for col in genre_schema:
    print(col)

Genre table schema:
(0, 'GenreId', 'INTEGER', 1, None, 1)
(1, 'Name', 'NVARCHAR(120)', 0, None, 0)


In [5]:
df_tracks = pd.read_sql_query("SELECT * FROM Track LIMIT 5;", conn)
df_tracks

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,NaN,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


In [6]:
df_genre = pd.read_sql_query("SELECT * FROM Genre LIMIT 5;", conn)
df_genre

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll


In [14]:
query = """
SELECT Genre.Name, AVG(Track.Milliseconds) AS AvgLength
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
GROUP BY Genre.Name
ORDER BY AvgLength DESC
LIMIT 1;
"""

pd.read_sql_query(query, conn)

,Name,AvgLength
0,Sci Fi & Fantasy,2.911783e+06


In [15]:
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    options={"temperature": 0}
)

print(response["message"]["content"])

Hello!


In [16]:
def call_llama(prompt, temperature=0):
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature}
    )
    return response["message"]["content"]

In [17]:
call_llama("What is sql agent")

'SQL Server Agent (SSA) is a component of Microsoft SQL Server that allows you to automate and manage the execution of stored procedures, jobs, and other tasks on your database. It provides a centralized management interface for scheduling, monitoring, and executing database maintenance tasks, such as backups, index updates, and data imports.\n\nWith SQL Server Agent, you can:\n\n1. **Schedule tasks**: Create schedules for tasks to run at specific times or intervals.\n2. **Run stored procedures**: Execute stored procedures, including those that perform database maintenance tasks.\n3. **Monitor jobs**: Track the status of running jobs and receive notifications when a job fails or completes successfully.\n4. **Manage connections**: Establish connections to remote servers and databases for job execution.\n5. **Create alerts**: Set up alerts to notify administrators when a job fails or completes unexpectedly.\n\nSQL Server Agent provides several benefits, including:\n\n1. **Improved produc

In [18]:
def get_schema_string(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    schema = ""

    for table in tables:
        table_name = table[0]
        schema += f"\nTable: {table_name}\n"

        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        for col in columns:
            schema += f"  - {col[1]} ({col[2]})\n"

    return schema


schema = get_schema_string(conn)
print(schema[:1000])  # preview first part


Table: Album
  - AlbumId (INTEGER)
  - Title (NVARCHAR(160))
  - ArtistId (INTEGER)

Table: Artist
  - ArtistId (INTEGER)
  - Name (NVARCHAR(120))

Table: Customer
  - CustomerId (INTEGER)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))
  - SupportRepId (INTEGER)

Table: Employee
  - EmployeeId (INTEGER)
  - LastName (NVARCHAR(20))
  - FirstName (NVARCHAR(20))
  - Title (NVARCHAR(30))
  - ReportsTo (INTEGER)
  - BirthDate (DATETIME)
  - HireDate (DATETIME)
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))

Table: Genre
  - GenreId (INTEGER)
  - Name (NVARCHAR(120))

Table: Invoice
  - InvoiceId

In [21]:
question = "Which genre has the highest average track length?"

prompt = f"""
You are a SQLite expert.

Database meaning:
- Track.Milliseconds represents the duration of a track.
- "Track length" means Track.Milliseconds.
- "Average track length" means AVG(Track.Milliseconds).

Rules:
1. Use ONLY table and column names exactly as given.
2. Do NOT invent tables or columns.
3. Join Track and Genre using Track.GenreId = Genre.GenreId.
4. Use GROUP BY when using aggregates like AVG().
5. Return ONLY valid SQLite SQL.
6. Do not explain anything.

Example:
Question: Which genre has the highest average track length?
SQL:
SELECT Genre.Name
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
GROUP BY Genre.Name
ORDER BY AVG(Track.Milliseconds) DESC
LIMIT 1;

Schema:
{schema}

Question:
{question}

SQL:
"""

In [23]:
sql_query = call_llama(prompt)
print(sql_query)

SELECT Genre.Name
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
GROUP BY Genre.Name
ORDER BY AVG(Track.Milliseconds) DESC
LIMIT 1;


In [24]:
result_df = pd.read_sql_query(sql_query, conn)
result_df


,Name
0,Sci Fi & Fantasy
